In [ ]:
#TODO: Set the simulation in hover for fixed time (10 seconds) in the starting point?
#TODO: Add wind disturbance to the simulation

In [39]:
from main import *
from Utils.plotting_functions import *
from Worlds.World import World
import Drone.Simulation
from matplotlib import pyplot as plt
from Optimizations.PSO_optimizer import PSOOptimizer
from Optimizations.optimizer import Optimizer as CostWrapper


waypoints_optimized_sac = [{"x": 27.664927049116653, "y": 88.29509735107422, "z": 10.923457145690918, "v": 18.322269439697266}, 
                           {"x": 37.11072626980868, "y": 85.373291015625, "z": 23.00494384765625, "v": 13.508941650390625}, 
                           {"x": 74.18110830133611, "y": 89.29581451416016, "z": 26.83894920349121, "v": 15.422500610351562}, 
                           {"x": 95.0, "y": 50.0, "z": 1.0, "v": 15.422500610351562}]

parameters = load_parameters("Settings/simulation_parameters.yaml")

A = parameters['start_point']
B = parameters['end_point']

init_state = create_initial_state(A[0], A[1], A[2])
thrust_max = get_max_thrust_from_rotor_model(parameters)


waypoints_selected = waypoints_optimized_sac

k_pid_pos = tuple(map(float, parameters['k_pid_pos']))
k_pid_alt = tuple(map(float, parameters['k_pid_alt']))
k_pid_att = tuple(map(float, parameters['k_pid_att']))
k_pid_yaw = tuple(map(float, parameters['k_pid_yaw']))
k_pid_hsp = tuple(map(float, parameters['k_pid_hsp']))
k_pid_vsp = tuple(map(float, parameters['k_pid_vsp']))

all_pid_list = [*k_pid_pos, *k_pid_alt, *k_pid_att, *k_pid_yaw, *k_pid_hsp, *k_pid_vsp]

In [40]:
def build_pid_gains(all_pid_list):
    return {
            'k_pid_pos': (all_pid_list[0], all_pid_list[1], all_pid_list[2]),
            'k_pid_alt': (all_pid_list[3], all_pid_list[4], all_pid_list[5]),
            'k_pid_att': (all_pid_list[6], all_pid_list[7], all_pid_list[8]),
            'k_pid_yaw': (all_pid_list[9], all_pid_list[10], all_pid_list[11]),
            'k_pid_hsp': (all_pid_list[12], all_pid_list[13], all_pid_list[14]),
            'k_pid_vsp': (all_pid_list[15], all_pid_list[16], all_pid_list[17]),
    }

In [41]:
world = World.load_world(parameters['world_data_path'])
noise_model = load_dnn_noise_model(parameters)

In [42]:
import pandas as pd

experiment_dataset = pd.DataFrame(
    columns=[
        'varied_pid_name',
        'k_pid_pos', 
        'k_pid_alt', 
        'k_pid_att', 
        'k_pid_yaw', 
        'k_pid_hsp', 
        'k_pid_vsp', 
        'time_history',
        'rpm_history',
        'rpm_max',
        'rpm_min',
        'rpm_mean',
        'rpm_std',
    ]
)

pid_gains = build_pid_gains(all_pid_list)
quad_controller = create_quadcopter_controller(init_state, pid_gains, thrust_max, parameters)
drone = create_quadcopter_model(init_state, quad_controller, parameters)
sim = Simulation(
    drone,
    world,
    waypoints_selected,
    dt=float(parameters['dt']),
    max_simulation_time=float(parameters['simulation_time']),
    frame_skip=int(parameters['frame_skip']),
    target_reached_threshold=float(parameters['threshold']),
    target_shift_threshold_distance=float(parameters['target_shift_threshold_distance']),
    noise_model=noise_model,
    generate_sound_emission_map=True,
    compute_psychoacoustics=False,
    noise_annoyance_radius=0,
)
sim.startSimulation()
experiment_dataset = pd.concat([experiment_dataset, pd.DataFrame({
    'varied_pid_name': [None],
    'k_pid_pos': [pid_gains['k_pid_pos']],
    'k_pid_alt': [pid_gains['k_pid_alt']],
    'k_pid_att': [pid_gains['k_pid_att']],
    'k_pid_yaw': [pid_gains['k_pid_yaw']],
    'k_pid_hsp': [pid_gains['k_pid_hsp']],
    'k_pid_vsp': [pid_gains['k_pid_vsp']],
    'time_history': [sim.time_history],
    'rpm_history': [sim.rpms_history],
    'rpm_max': [np.max(sim.rpms_history)],
    'rpm_min': [np.min(sim.rpms_history)],
    'rpm_mean': [np.mean(sim.rpms_history)],
    'rpm_std': [np.std(sim.rpms_history)],
}, index=[0])], ignore_index=True)

Final target reached at time: 14.74 s
Simulation completed in 3.39 seconds.


In [43]:
from tqdm import tqdm

prc_var = 0.1

for i in tqdm(range(len(all_pid_list)), total=len(all_pid_list)):
    perturbed_pid_list = all_pid_list.copy()

    perturbation = all_pid_list[i] * prc_var
    perturbed_pid_list[i] += perturbation

    pid_gains = build_pid_gains(perturbed_pid_list)
    quad_controller = create_quadcopter_controller(init_state, pid_gains, thrust_max, parameters)
    drone = create_quadcopter_model(init_state, quad_controller, parameters)
    sim = Simulation(
        drone,
        world,
        waypoints_selected,
        dt=float(parameters['dt']),
        max_simulation_time=float(parameters['simulation_time']),
        frame_skip=int(parameters['frame_skip']),
        target_reached_threshold=float(parameters['threshold']),
        target_shift_threshold_distance=float(parameters['target_shift_threshold_distance']),
        noise_model=noise_model,
        generate_sound_emission_map=True,
        compute_psychoacoustics=False,
        noise_annoyance_radius=0,
    )
    sim.startSimulation(verbose=False)
    time_history = sim.time_history
    rpm_history = sim.rpms_history
    rpm_max = np.max(rpm_history)
    rpm_min = np.min(rpm_history)
    rpm_mean = np.mean(rpm_history)
    rpm_std = np.std(rpm_history)

    pid_set_name = list(pid_gains.keys())[i // 3]
    pid_term = 'P' if i % 3 == 0 else ('I' if i % 3 == 1 else 'D')

    experiment_dataset = pd.concat([experiment_dataset, pd.DataFrame({   
            'varied_pid_name': [pid_set_name + '_' + pid_term],
            'k_pid_pos': [pid_gains['k_pid_pos']], 
            'k_pid_alt': [pid_gains['k_pid_alt']],
            'k_pid_att': [pid_gains['k_pid_att']],
            'k_pid_yaw': [pid_gains['k_pid_yaw']],
            'k_pid_hsp': [pid_gains['k_pid_hsp']],
            'k_pid_vsp': [pid_gains['k_pid_vsp']],
            'time_history': [time_history],
            'rpm_history': [rpm_history],
            'rpm_max': [rpm_max],
            'rpm_min': [rpm_min],
            'rpm_mean': [rpm_mean],
            'rpm_std': [rpm_std],
        }
    )], ignore_index=True)

100%|██████████| 18/18 [01:01<00:00,  3.44s/it]


In [44]:
display(experiment_dataset)

,varied_pid_name,k_pid_pos,k_pid_alt,k_pid_att,k_pid_yaw,k_pid_hsp,k_pid_vsp,time_history,rpm_history,rpm_max,rpm_min,rpm_mean,rpm_std
0,None,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1628.685226,757.009956
1,k_pid_pos_P,"(1.128812721359806, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1629.519492,761.051244
2,k_pid_pos_I,"(1.026193383054369, 0.0369497837668771, 0.3053...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1627.260485,759.367627
3,k_pid_pos_D,"(1.026193383054369, 0.03359071251534282, 0.335...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1627.415790,759.712367
4,k_pid_alt_P,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.8014958751426673, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1628.376553,761.695144
5,k_pid_alt_I,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1628.685226,757.009956
6,k_pid_alt_D,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.022969196847979254)","(8.967805799620551, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.40120119426...",3000.0,0.0,1627.335872,760.131895
7,k_pid_att_P,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(9.864586379582606, 3.629835988855863, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2340.74540667977...",3000.0,0.0,1629.092003,756.518120
8,k_pid_att_I,"(1.026193383054369, 0.03359071251534282, 0.305...","(1.6377235228569702, 0.0, 0.020881088043617504)","(8.967805799620551, 3.992819587741449, 0.56690...","(0.5, 1e-06, 0.1)","(0.21751398283713122, 0.011190626363252196, 0.0)","(249.7994300660337, 129.48998830292263, 0.1614...","[0.0, 0.008, 0.016, 0.024, 0.032, 0.04, 0.048,...","[[0.0, 3000.0, 3000.0, 0.0], [2358.27862729139...",3000.0,0.0,1627.722324,760.096878
9,k_pid_att

In [ ]:
# Save the dataset to a CSV file separated by semicolons
experiment_dataset.to_csv('pid_gains_sensitivity_analysis_dataset.csv', sep=';', index=False)